# Specialty Signal Methods Test

### Purpose
- reshape data
- prepare comparable indicators
- calculate specialty summaries
- option 1 = rank method
- option 2 = PCA (HOLD FOR NOW)
- option 3 = equal-weight composite (maybe change weighting in future)
- compare results

In [154]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

INPUT_DIR = Path("../Inputs")
OUTPUT_DIR = Path("../outputs")
METHODS_OUTPUT_DIR = OUTPUT_DIR / "Methods_Test"

METHODS_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MASTER_FILE = OUTPUT_DIR / "Specialty_Master.xlsx"
CONFIG_FILE = INPUT_DIR / "indicator_config.xlsx"

EXPECTED_ICBS = 42
# to flag in the coverage report if not all ICBs have data
COVERAGE_THRESHOLD = 100

specialty_master = pd.read_excel(MASTER_FILE)
indicator_config = pd.read_excel(CONFIG_FILE)

indicator_config["Include"] = (
    indicator_config["Include"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
)

active_config = indicator_config.loc[
    indicator_config["Include"]
].copy()

print("Master rows:", len(specialty_master))
print(
    "Specialties:",
    active_config["Specialty"].nunique()
)
print(
    "Configured indicators:",
    len(active_config)
)

Master rows: 42
Specialties: 4
Configured indicators: 16


#### Part A: prepare the methods dataset

In [155]:
identifier_columns = [
    "ICB_Geography_Code",
    "ICB_Code",
    "ICB_Name",
    "Population_2024",
    "IMD_Average_Score"
]

indicator_columns = [
    indicator
    for indicator in active_config["Indicator"]
    if indicator in specialty_master.columns
]

signal_long = specialty_master.melt(
    id_vars=identifier_columns,
    value_vars=indicator_columns,
    var_name="Indicator",
    value_name="Raw_Value"
)

config_fields = [
    "Specialty",
    "Domain",
    "Indicator",
    "Direction",
    "Measure_Family",
    "Unit",
    "Value_Type",
    "Preparation_Method",
    "Denominator_Column",
    "Aggregation_Method",
    "Include_In_Composite",
    "Missing_Value_Rule",
    "Weight"
]

signal_long = signal_long.merge(
    active_config[config_fields],
    on="Indicator",
    how="left",
    validate="many_to_one"
)

signal_long["Raw_Value"] = pd.to_numeric(
    signal_long["Raw_Value"],
    errors="coerce"
)

In [156]:
#create an analysis value

signal_long["Analysis_Value"] = signal_long[
    "Raw_Value"
].copy()

rate_mask = signal_long[
    "Preparation_Method"
].eq("Rate_Per_100000_Population")

valid_population = (
    signal_long["Population_2024"]
    .gt(0)
)

signal_long.loc[
    rate_mask & valid_population,
    "Analysis_Value"
] = (
    signal_long.loc[
        rate_mask & valid_population,
        "Raw_Value"
    ]
    .div(
        signal_long.loc[
            rate_mask & valid_population,
            "Population_2024"
        ]
    )
    .mul(100_000)
)

In [157]:
signal_long["Analysis_Unit"] = (
    signal_long["Unit"]
)

signal_long.loc[
    rate_mask,
    "Analysis_Unit"
] = "Rate per 100,000 ICB population"

In [158]:
#QA the report generated

qa_report = (
    signal_long
    .groupby(
        [
            "Specialty",
            "Measure_Family",
            "Indicator"
        ],
        as_index=False
    )
    .agg(
        ICBs_Available=(
            "Analysis_Value",
            "count"
        ),
        Missing_Values=(
            "Analysis_Value",
            lambda x: x.isna().sum()
        ),
        Zero_Values=(
            "Analysis_Value",
            lambda x: x.eq(0).sum()
        ),
        Minimum=(
            "Analysis_Value",
            "min"
        ),
        Maximum=(
            "Analysis_Value",
            "max"
        )
    )
)

qa_report["Coverage_Percent"] = (
    qa_report["ICBs_Available"]
    .div(EXPECTED_ICBS)
    .mul(100)
)

display(qa_report.round(3))

,Specialty,Measure_Family,Indicator,ICBs_Available,Missing_Values,Zero_Values,Minimum,Maximum,Coverage_Percent
0,Cardiovascular,Elective_LOS_6plus,Cardiology_ElectiveAdmissions_6day+,42,0,4,0.000,37.500,100.000
1,Cardiovascular,Emergency_LOS_6plus,Cardiology_EmergencyAdmissions_6day+,42,0,0,14.600,38.200,100.000
2,Cardiovascular,RTT_18plus,Cardiology_RTT_18plus,42,0,0,16.740,53.030,100.000
3,Cardiovascular,RTT_Incomplete,Cardiology_RTT_Incomplete,42,0,0,298.796,1300.606,100.000
4,Gynaecology,Elective_LOS_6plus,Gynaecology_ElectiveAdmissions_6day+,42,0,6,0.000,7.400,100.000
5,Gynaecology,Emergency_LOS_6plus,Gynaecology_EmergencyAdmissions_6day+,42,0,0,0.400,6.200,100.000
6,Gynaecology,RTT_18plus,Gynaecology_RTT_18plus,42,0,0,28.880,55.880,100.000
7,Gynaecology,RTT_Incomplete,Gynaecology_RTT_Incomplete,42,0,0,441.499,1462.479,100.000
8,Respiratory,Elective_LOS_6plus,Respiratory_ElectiveAdmissions_6day+,41,1,9,0.000,75.000,97.619
9,Respiratory,Emergency_LOS_6plus,Respiratory_EmergencyAdmissions_6day+,42,0,0,23.800,66.700,100.000


#### Part B: create the specialty-level input

In [159]:
# summarise each specialty and matched indicator family

specialty_indicator_summary = (
    signal_long
    .groupby(
        [
            "Specialty",
            "Domain",
            "Measure_Family",
            "Indicator"
        ],
        as_index=False
    )
    .agg(
        Specialty_Value=(
            "Analysis_Value",
            "median"
        ),
        ICB_Mean=(
            "Analysis_Value",
            "mean"
        ),
        ICB_Median=(
            "Analysis_Value",
            "median"
        ),
        ICB_P10=(
            "Analysis_Value",
            lambda x: x.quantile(0.10)
        ),
        ICB_P90=(
            "Analysis_Value",
            lambda x: x.quantile(0.90)
        ),
        ICBs_Available=(
            "Analysis_Value",
            "count"
        )
    )
)

specialty_indicator_summary[
    "P90_P10_Variation"
] = (
    specialty_indicator_summary["ICB_P90"]
    - specialty_indicator_summary["ICB_P10"]
)

display(
    specialty_indicator_summary.round(3)
)

,Specialty,Domain,Measure_Family,Indicator,Specialty_Value,ICB_Mean,ICB_Median,ICB_P10,ICB_P90,ICBs_Available,P90_P10_Variation
0,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,38.600,37.521,38.600,26.447,47.865,42,21.418
1,Cardiovascular,Hospital Burden,Elective_LOS_6plus,Cardiology_ElectiveAdmissions_6day+,10.450,12.214,10.450,3.010,24.810,42,21.800
2,Cardiovascular,Hospital Burden,Emergency_LOS_6plus,Cardiology_EmergencyAdmissions_6day+,26.850,26.883,26.850,18.330,34.510,42,16.180
3,Cardiovascular,Scale of Pressure,RTT_Incomplete,Cardiology_RTT_Incomplete,674.294,693.361,674.294,373.107,1027.971,42,654.864
4,Gynaecology,Access Pressure,RTT_18plus,Gynaecology_RTT_18plus,45.355,44.349,45.355,35.294,52.059,42,16.765
5,Gynaecology,Hospital Burden,Elective_LOS_6plus,Gynaecology_ElectiveAdmissions_6day+,1.600,1.898,1.600,0.000,4.120,42,4.120
6,Gynaecology,Hospital Burden,Emergency_LOS_6plus,Gynaecology_EmergencyAdmissions_6day+,2.300,2.548,2.300,0.930,4.490,42,3.560
7,Gynaecology,Scale of Pressure,RTT_Incomplete,Gynaecology_RTT_Incomplete,845.055,919.089,845.055,596.092,1266.741,42,670.649
8,Respiratory,Access Pressure,RTT_18plus,Respiratory_RTT_18plus,36.555,35.012,36.555,20.108,45.907,42,25.799
9,Respiratory,Hospital Burden,Elective_LOS_6plus,Respiratory_ElectiveAdmissions_6day+,15.800,17.832,15.800,0.000,35.700,41,35.700


## OPTION 1 TESTING - Rank Method

For every matched measure family: 
- compare specialty values
- rank the specialites
- make sure higher values represent stronger pressure signal
- calculate the mean rank for each specialty

Because higher is worse for all current indicators, the highest value receives rank 1.

In [160]:
rank_results = (
    specialty_indicator_summary.copy()
)

rank_results["Indicator_Rank"] = (
    rank_results
    .groupby("Measure_Family")[
        "Specialty_Value"
    ]
    .rank(
        ascending=False,
        method="average"
    )
)

In [161]:
rank_specialty_summary = (
    rank_results
    .groupby(
        "Specialty",
        as_index=False
    )
    .agg(
        Mean_Rank=(
            "Indicator_Rank",
            "mean"
        ),
        Best_Rank=(
            "Indicator_Rank",
            "min"
        ),
        Worst_Rank=(
            "Indicator_Rank",
            "max"
        ),
        Indicators_Included=(
            "Measure_Family",
            "nunique"
        )
    )
    .sort_values(
        "Mean_Rank",
        ascending=True
    )
)

rank_specialty_summary[
    "Overall_Rank"
] = (
    rank_specialty_summary[
        "Mean_Rank"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

display(rank_specialty_summary)

,Specialty,Mean_Rank,Best_Rank,Worst_Rank,Indicators_Included,Overall_Rank
3,Trauma and Orthopaedics,1.50,1.0,3.0,4,1.0
0,Cardiovascular,2.75,2.0,3.0,4,2.0
2,Respiratory,2.75,1.0,4.0,4,2.0
1,Gynaecology,3.00,2.0,4.0,4,4.0


In [162]:
#create a transparent wide-table with all the workings/detail

rank_component_table = (
    rank_results
    .pivot(
        index="Specialty",
        columns="Measure_Family",
        values="Indicator_Rank"
    )
    .reset_index()
)

rank_component_table = (
    rank_component_table
    .merge(
        rank_specialty_summary,
        on="Specialty",
        how="left",
        validate="one_to_one"
    )
)

display(rank_component_table)

,Specialty,Elective_LOS_6plus,Emergency_LOS_6plus,RTT_18plus,RTT_Incomplete,Mean_Rank,Best_Rank,Worst_Rank,Indicators_Included,Overall_Rank
0,Cardiovascular,2.0,3.0,3.0,3.0,2.75,2.0,3.0,4,2.0
1,Gynaecology,4.0,4.0,2.0,2.0,3.00,2.0,4.0,4,4.0
2,Respiratory,1.0,2.0,4.0,4.0,2.75,1.0,4.0,4,2.0
3,Trauma and Orthopaedics,3.0,1.0,1.0,1.0,1.50,1.0,3.0,4,1.0


Lower mean rank (e.g. 1) means stronger overall relative signal
Don't report only on final rank - retain individual measure-family ranks to understand the detail behind

## OPTION 2 TESTING = PCA
PCA uses one row for every ICB-specialty combination.
The four matched measure families become columns:
- RTT over 18 weeks
- RTT incomplete pathways per 100,000 population
- emergency LOS over 6 days
- elective LOS over 6 days

PCA is exploratory. PC1 will only be treated as a possible overall signal if the component loadings are coherent and substantively interpretable.

In [163]:
# ---------------------------------------------------------
# 1. Create the ICB-specialty PCA input table
# ---------------------------------------------------------

pca_input = (
    signal_long
    .pivot_table(
        index=[
            "ICB_Geography_Code",
            "ICB_Code",
            "ICB_Name",
            "Specialty"
        ],
        columns="Measure_Family",
        values="Analysis_Value",
        aggfunc="first"
    )
    .reset_index()
)

# Remove the column-axis label created by pivot_table
pca_input.columns.name = None


pca_feature_columns = [
    "RTT_18plus",
    "RTT_Incomplete",
    "Emergency_LOS_6plus",
    "Elective_LOS_6plus"
]


missing_pca_columns = [
    column
    for column in pca_feature_columns
    if column not in pca_input.columns
]

if missing_pca_columns:
    raise ValueError(
        "The following PCA features are missing: "
        f"{missing_pca_columns}"
    )


print(
    f"PCA input rows: {len(pca_input)}"
)

print(
    "Expected maximum rows:",
    EXPECTED_ICBS
    * active_config["Specialty"].nunique()
)

display(
    pca_input.head()
)

PCA input rows: 168
Expected maximum rows: 168


,ICB_Geography_Code,ICB_Code,ICB_Name,Specialty,Elective_LOS_6plus,Emergency_LOS_6plus,RTT_18plus,RTT_Incomplete
0,E54000008,QYG,Cheshire And Merseyside ICB,Cardiovascular,7.8,21.9,37.89,758.920634
1,E54000008,QYG,Cheshire And Merseyside ICB,Gynaecology,1.5,2.8,45.95,1313.400308
2,E54000008,QYG,Cheshire And Merseyside ICB,Respiratory,13.6,35.5,36.45,438.896164
3,E54000008,QYG,Cheshire And Merseyside ICB,Trauma and Orthopaedics,6.4,49.3,44.98,986.225948
4,E54000010,QNC,Staffordshire And Stoke-On-Trent ICB,Cardiovascular,17.4,18.6,38.28,365.411039


In [ ]:
# inspect missing values before PCA

pca_missing_summary = (
    pca_input[
        pca_feature_columns
    ]
    .isna()
    .sum()
    .reset_index()
)

pca_missing_summary.columns = [
    "Measure_Family",
    "Missing_Values"
]

display(pca_missing_summary)

,Measure_Family,Missing_Values
0,RTT_18plus,0
1,RTT_Incomplete,0
2,Emergency_LOS_6plus,0
3,Elective_LOS_6plus,1


In [165]:
# impute the missing value transparently
# use median for the same specialty and measure family - meaning rather than the value missing being just deleted or using 0, it just uses the median value for that

pca_model_data = pca_input.copy()

pca_imputation_report = []

for feature in pca_feature_columns:

    missing_before = (
        pca_model_data[feature]
        .isna()
    )

    if missing_before.any():

        specialty_medians = (
            pca_model_data
            .groupby("Specialty")[feature]
            .transform("median")
        )

        pca_model_data.loc[
            missing_before,
            feature
        ] = specialty_medians.loc[
            missing_before
        ]

        for row_index in pca_model_data.index[
            missing_before
        ]:

            pca_imputation_report.append({
                "ICB_Code":
                    pca_model_data.loc[
                        row_index,
                        "ICB_Code"
                    ],
                "ICB_Name":
                    pca_model_data.loc[
                        row_index,
                        "ICB_Name"
                    ],
                "Specialty":
                    pca_model_data.loc[
                        row_index,
                        "Specialty"
                    ],
                "Measure_Family":
                    feature,
                "Imputed_Value":
                    pca_model_data.loc[
                        row_index,
                        feature
                    ],
                "Imputation_Method":
                    "Specialty median"
            })


pca_imputation_report = pd.DataFrame(
    pca_imputation_report
)

display(
    pca_imputation_report
)


remaining_missing_values = (
    pca_model_data[
        pca_feature_columns
    ]
    .isna()
    .sum()
    .sum()
)

if remaining_missing_values > 0:
    raise ValueError(
        "Missing PCA feature values remain after "
        f"imputation: {remaining_missing_values}"
    )

,ICB_Code,ICB_Name,Specialty,Measure_Family,Imputed_Value,Imputation_Method
0,QHG,"Bedfordshire, Luton And Milton Keynes ICB",Respiratory,Elective_LOS_6plus,15.8,Specialty median


In [166]:
# standardise the four indicators
# because 3 are percentages and 1 is a rate per 100,000

pca_scaler = StandardScaler()

pca_standardised_values = (
    pca_scaler.fit_transform(
        pca_model_data[
            pca_feature_columns
        ]
    )
)

pca_standardised = pd.DataFrame(
    pca_standardised_values,
    columns=pca_feature_columns,
    index=pca_model_data.index
)

# Retain identifiers for checking and interpretation
pca_standardised_output = pd.concat(
    [
        pca_model_data[
            [
                "ICB_Geography_Code",
                "ICB_Code",
                "ICB_Name",
                "Specialty"
            ]
        ],
        pca_standardised.add_suffix(
            "_Standardised"
        )
    ],
    axis=1
)

display(
    pca_standardised_output.head()
)

,ICB_Geography_Code,ICB_Code,ICB_Name,Specialty,RTT_18plus_Standardised,RTT_Incomplete_Standardised,Emergency_LOS_6plus_Standardised,Elective_LOS_6plus_Standardised
0,E54000008,QYG,Cheshire And Merseyside ICB,Cardiovascular,-0.320265,-0.094449,-0.366041,-0.175587
1,E54000008,QYG,Cheshire And Merseyside ICB,Gynaecology,0.522146,1.087138,-1.453890,-0.746880
2,E54000008,QYG,Cheshire And Merseyside ICB,Respiratory,-0.470771,-0.776416,0.408554,0.350365
3,E54000008,QYG,Cheshire And Merseyside ICB,Trauma and Orthopaedics,0.420764,0.389935,1.194539,-0.302541
4,E54000010,QNC,Staffordshire And Stoke-On-Trent ICB,Cardiovascular,-0.279504,-0.933011,-0.553994,0.694955


In [167]:
# expect to see means close to 0 and standard deviation close to 1

pca_scaling_check = pd.DataFrame({
    "Standardised_Mean":
        pca_standardised.mean(),
    "Standardised_Standard_Deviation":
        pca_standardised.std(
            ddof=0
        )
}).reset_index()

pca_scaling_check = (
    pca_scaling_check.rename(
        columns={
            "index":
                "Measure_Family"
        }
    )
)

display(
    pca_scaling_check.round(3)
)

,Measure_Family,Standardised_Mean,Standardised_Standard_Deviation
0,RTT_18plus,0.0,1.0
1,RTT_Incomplete,-0.0,1.0
2,Emergency_LOS_6plus,0.0,1.0
3,Elective_LOS_6plus,0.0,1.0


In [168]:
# dont initially write to 1 vector/compenent as suggested - first check if one componenet is sufficient

pca_full = PCA(
    n_components=len(
        pca_feature_columns
    )
)

pca_component_scores = (
    pca_full.fit_transform(
        pca_standardised
    )
)


pca_component_names = [
    f"PC{component_number}"
    for component_number in range(
        1,
        len(pca_feature_columns) + 1
    )
]


pca_scores = pd.DataFrame(
    pca_component_scores,
    columns=pca_component_names,
    index=pca_model_data.index
)


pca_scores = pd.concat(
    [
        pca_model_data[
            [
                "ICB_Geography_Code",
                "ICB_Code",
                "ICB_Name",
                "Specialty"
            ]
        ],
        pca_scores
    ],
    axis=1
)


display(
    pca_scores.head()
)

,ICB_Geography_Code,ICB_Code,ICB_Name,Specialty,PC1,PC2,PC3,PC4
0,E54000008,QYG,Cheshire And Merseyside ICB,Cardiovascular,0.022756,-0.507880,-0.056128,0.121231
1,E54000008,QYG,Cheshire And Merseyside ICB,Gynaecology,1.825605,-0.577495,0.285140,0.615219
2,E54000008,QYG,Cheshire And Merseyside ICB,Respiratory,-1.012358,-0.131107,-0.032031,-0.266522
3,E54000008,QYG,Cheshire And Merseyside ICB,Trauma and Orthopaedics,0.134509,1.055432,-0.680383,-0.502590
4,E54000010,QNC,Staffordshire And Stoke-On-Trent ICB,Cardiovascular,-0.822503,-0.622774,0.819148,0.056060


In [169]:
# Examine explained variance

pca_explained_variance = pd.DataFrame({
    "Component":
        pca_component_names,
    "Explained_Variance_Ratio":
        pca_full.explained_variance_ratio_
})


pca_explained_variance[
    "Explained_Variance_Percent"
] = (
    pca_explained_variance[
        "Explained_Variance_Ratio"
    ]
    .mul(100)
)


pca_explained_variance[
    "Cumulative_Variance_Percent"
] = (
    pca_explained_variance[
        "Explained_Variance_Percent"
    ]
    .cumsum()
)


display(
    pca_explained_variance.round(2)
)

,Component,Explained_Variance_Ratio,Explained_Variance_Percent,Cumulative_Variance_Percent
0,PC1,0.45,44.64,44.64
1,PC2,0.28,28.04,72.69
2,PC3,0.15,15.06,87.75
3,PC4,0.12,12.25,100.00


In [170]:
# Examine PCA loadings

pca_loadings = pd.DataFrame(
    pca_full.components_.T,
    index=pca_feature_columns,
    columns=pca_component_names
)

pca_loadings = (
    pca_loadings
    .reset_index()
    .rename(
        columns={
            "index":
                "Measure_Family"
        }
    )
)


display(
    pca_loadings.round(3)
)

,Measure_Family,PC1,PC2,PC3,PC4
0,RTT_18plus,0.502,0.485,0.562,-0.445
1,RTT_Incomplete,0.553,0.403,-0.323,0.654
2,Emergency_LOS_6plus,-0.382,0.677,-0.521,-0.351
3,Elective_LOS_6plus,-0.545,0.379,0.556,0.501


In [171]:
# Orient PC1 so higher represents a stronger signal

pc1_loadings = pca_loadings.set_index(
    "Measure_Family"
)["PC1"]


if pc1_loadings.mean() < 0:

    pca_scores["PC1"] = (
        pca_scores["PC1"]
        * -1
    )

    pca_loadings["PC1"] = (
        pca_loadings["PC1"]
        * -1
    )

    print(
        "PC1 sign reversed so that higher values "
        "represent a stronger overall signal."
    )

else:

    print(
        "PC1 orientation retained. Higher values "
        "already represent a stronger overall signal."
    )


display(
    pca_loadings.round(3)
)

PC1 orientation retained. Higher values already represent a stronger overall signal.


,Measure_Family,PC1,PC2,PC3,PC4
0,RTT_18plus,0.502,0.485,0.562,-0.445
1,RTT_Incomplete,0.553,0.403,-0.323,0.654
2,Emergency_LOS_6plus,-0.382,0.677,-0.521,-0.351
3,Elective_LOS_6plus,-0.545,0.379,0.556,0.501


In [172]:
# 9. Summarise PC1 by specialty

pca_specialty_summary = (
    pca_scores
    .groupby(
        "Specialty",
        as_index=False
    )
    .agg(
        PCA_PC1_Mean=(
            "PC1",
            "mean"
        ),
        PCA_PC1_Median=(
            "PC1",
            "median"
        ),
        PCA_PC1_Minimum=(
            "PC1",
            "min"
        ),
        PCA_PC1_Maximum=(
            "PC1",
            "max"
        ),
        ICBs_Included=(
            "ICB_Code",
            "nunique"
        )
    )
)


# Use the median PC1 as the primary specialty summary,
# consistent with the baseline rank and composite methods

pca_specialty_summary[
    "PCA_Specialty_Value"
] = (
    pca_specialty_summary[
        "PCA_PC1_Median"
    ]
)


pca_specialty_summary[
    "PCA_Rank"
] = (
    pca_specialty_summary[
        "PCA_Specialty_Value"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


pca_specialty_summary = (
    pca_specialty_summary
    .sort_values(
        "PCA_Rank"
    )
    .reset_index(drop=True)
)


display(
    pca_specialty_summary.round(3)
)

,Specialty,PCA_PC1_Mean,PCA_PC1_Median,PCA_PC1_Minimum,PCA_PC1_Maximum,ICBs_Included,PCA_Specialty_Value,PCA_Rank
0,Gynaecology,1.263,1.284,0.007,2.252,42,1.284,1.0
1,Trauma and Orthopaedics,0.626,0.497,-0.710,4.436,42,0.497,2.0
2,Cardiovascular,-0.400,-0.327,-3.146,1.208,42,-0.327,3.0
3,Respiratory,-1.489,-1.350,-3.643,0.088,42,-1.350,4.0


In [173]:
# Inspect PC1 and PC2 by specialty

pca_two_component_summary = (
    pca_scores
    .groupby(
        "Specialty",
        as_index=False
    )
    .agg(
        Median_PC1=(
            "PC1",
            "median"
        ),
        Median_PC2=(
            "PC2",
            "median"
        ),
        Mean_PC1=(
            "PC1",
            "mean"
        ),
        Mean_PC2=(
            "PC2",
            "mean"
        )
    )
)


display(
    pca_two_component_summary.round(3)
)

,Specialty,Median_PC1,Median_PC2,Mean_PC1,Mean_PC2
0,Cardiovascular,-0.327,-0.275,-0.400,-0.239
1,Gynaecology,1.284,-0.998,1.263,-0.993
2,Respiratory,-1.350,-0.182,-1.489,0.045
3,Trauma and Orthopaedics,0.497,1.120,0.626,1.187


## OPTION 3 TESTING = equal-weight composite

- get one specialty value per measure family;
- scale the specialties against one another within each family;
- average the scaled measure-family scores.

In [174]:
# create 0-100 scores
# lowest specialty = 0, highest = 100, other specialty between those two

def min_max_signal(series):
    minimum = series.min()
    maximum = series.max()

    if pd.isna(minimum) or pd.isna(maximum):
        return pd.Series(
            np.nan,
            index=series.index
        )

    if maximum == minimum:
        return pd.Series(
            50.0,
            index=series.index
        )

    return (
        series
        .sub(minimum)
        .div(maximum - minimum)
        .mul(100)
    )

In [175]:
#apply the scoring within each comparable measure family

composite_results = (
    specialty_indicator_summary.copy()
)

composite_results[
    "Standardised_Signal_0_100"
] = (
    composite_results
    .groupby(
        "Measure_Family",
        group_keys=False
    )["Specialty_Value"]
    .apply(min_max_signal)
)

In [176]:
# merge the configured weights

weight_lookup = (
    active_config[
        [
            "Indicator",
            "Weight",
            "Include_In_Composite"
        ]
    ]
    .copy()
)

weight_lookup[
    "Include_In_Composite"
] = (
    weight_lookup[
        "Include_In_Composite"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
)

composite_results = (
    composite_results
    .merge(
        weight_lookup,
        on="Indicator",
        how="left",
        validate="one_to_one"
    )
)

composite_results = composite_results.loc[
    composite_results[
        "Include_In_Composite"
    ]
].copy()

In [177]:
#validate the weights within each specialty
# each specialty should total 1.0 if using four 0.25 weightings (as we are currently - can change)

weight_check = (
    composite_results
    .groupby(
        "Specialty",
        as_index=False
    )
    .agg(
        Weight_Total=(
            "Weight",
            "sum"
        ),
        Included_Indicators=(
            "Measure_Family",
            "nunique"
        )
    )
)

display(weight_check)

,Specialty,Weight_Total,Included_Indicators
0,Cardiovascular,1.0,4
1,Gynaecology,1.0,4
2,Respiratory,1.0,4
3,Trauma and Orthopaedics,1.0,4


In [178]:
# calculate the compositve

composite_results[
    "Weighted_Contribution"
] = (
    composite_results[
        "Standardised_Signal_0_100"
    ]
    * composite_results["Weight"]
)

composite_specialty_summary = (
    composite_results
    .groupby(
        "Specialty",
        as_index=False
    )
    .agg(
        Composite_Signal_0_100=(
            "Weighted_Contribution",
            "sum"
        ),
        Indicators_Included=(
            "Measure_Family",
            "nunique"
        ),
        Weight_Total=(
            "Weight",
            "sum"
        )
    )
    .sort_values(
        "Composite_Signal_0_100",
        ascending=False
    )
)

composite_specialty_summary[
    "Composite_Rank"
] = (
    composite_specialty_summary[
        "Composite_Signal_0_100"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

display(
    composite_specialty_summary.round(2)
)

,Specialty,Composite_Signal_0_100,Indicators_Included,Weight_Total,Composite_Rank
3,Trauma and Orthopaedics,89.14,4,1.0,1.0
1,Gynaecology,54.40,4,1.0,2.0
0,Cardiovascular,42.24,4,1.0,3.0
2,Respiratory,31.17,4,1.0,4.0


In [179]:
# retain the contributions - so we can see the breakdown and understand detail

composite_component_table = (
    composite_results
    .pivot(
        index="Specialty",
        columns="Measure_Family",
        values="Standardised_Signal_0_100"
    )
    .reset_index()
    .merge(
        composite_specialty_summary,
        on="Specialty",
        how="left",
        validate="one_to_one"
    )
)

display(
    composite_component_table.round(2)
)

,Specialty,Elective_LOS_6plus,Emergency_LOS_6plus,RTT_18plus,RTT_Incomplete,Composite_Signal_0_100,Indicators_Included,Weight_Total,Composite_Rank
0,Cardiovascular,62.32,58.73,22.35,43.82,42.24,4,1.0,3.0
1,Gynaecology,0.00,0.00,96.17,67.03,54.40,4,1.0,2.0
2,Respiratory,100.00,86.96,0.00,0.00,31.17,4,1.0,4.0
3,Trauma and Orthopaedics,34.86,100.00,100.00,100.00,89.14,4,1.0,1.0


# COMPARE Options 1, 2 and 3

In [180]:
# ---------------------------------------------------------
# Compare Options 1, 2 and 3
# ---------------------------------------------------------

method_comparison = (
    rank_specialty_summary[
        [
            "Specialty",
            "Overall_Rank",
            "Mean_Rank"
        ]
    ]
    .merge(
        pca_specialty_summary[
            [
                "Specialty",
                "PCA_Specialty_Value",
                "PCA_Rank"
            ]
        ],
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        composite_specialty_summary[
            [
                "Specialty",
                "Composite_Rank",
                "Composite_Signal_0_100"
            ]
        ],
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
)


method_comparison[
    "Rank_Composite_Agree"
] = (
    method_comparison[
        "Overall_Rank"
    ]
    .eq(
        method_comparison[
            "Composite_Rank"
        ]
    )
)


method_comparison[
    "All_Three_Agree"
] = (
    method_comparison[
        "Overall_Rank"
    ]
    .eq(
        method_comparison[
            "Composite_Rank"
        ]
    )
    &
    method_comparison[
        "Overall_Rank"
    ]
    .eq(
        method_comparison[
            "PCA_Rank"
        ]
    )
)


display(
    method_comparison
    .sort_values(
        "Composite_Rank"
    )
    .round(3)
)

,Specialty,Overall_Rank,Mean_Rank,PCA_Specialty_Value,PCA_Rank,Composite_Rank,Composite_Signal_0_100,Rank_Composite_Agree,All_Three_Agree
3,Trauma and Orthopaedics,1.0,1.50,0.497,2.0,1.0,89.141,True,False
1,Gynaecology,4.0,3.00,1.284,1.0,2.0,54.396,False,False
0,Cardiovascular,2.0,2.75,-0.327,3.0,3.0,42.236,False,False
2,Respiratory,2.0,2.75,-1.350,4.0,4.0,31.167,False,False


## Sensitivity Analysis

The primary analysis used median ICB values and equal indicator weighting. Additional sensitivity analyses tested alternative aggregation, domain-equal weighting and removal of individual indicators.
KEEP CONFIG DOC THE SAME - dont alter the baseline (which is equal weighting, median, all indicators etc)

In [181]:
# =========================================================
# SENSITIVITY ANALYSIS
# =========================================================

def run_signal_scenario(
    specialty_inputs,
    scenario_name,
    value_column="ICB_Median",
    weight_mapping=None,
    excluded_measure_family=None
):
    """
    Run rank and composite methods under an alternative
    analytical scenario.

    Parameters
    ----------
    specialty_inputs:
        The specialty_indicator_summary dataframe.

    scenario_name:
        Descriptive name retained in the output.

    value_column:
        Column used as the specialty-level value.
        Expected examples:
        - ICB_Median
        - ICB_Mean

    weight_mapping:
        Dictionary assigning a weight to each measure family.
        If None, equal indicator weights are applied.

    excluded_measure_family:
        Optional measure family to remove from the scenario.

    Returns
    -------
    scenario_detail:
        Indicator-level results.

    scenario_summary:
        Specialty-level rank and composite results.
    """

    scenario_detail = specialty_inputs.copy()

    # Remove one measure family if requested
    if excluded_measure_family is not None:
        scenario_detail = scenario_detail.loc[
            ~scenario_detail[
                "Measure_Family"
            ].eq(excluded_measure_family)
        ].copy()

    # Validate the selected value column
    if value_column not in scenario_detail.columns:
        raise ValueError(
            f"{value_column} is not available. "
            f"Available columns are: "
            f"{scenario_detail.columns.tolist()}"
        )

    # Use either the mean or median as the specialty value
    scenario_detail["Scenario_Value"] = (
        scenario_detail[value_column]
    )

    # -----------------------------------------------------
    # Rank method
    # -----------------------------------------------------

    scenario_detail["Indicator_Rank"] = (
        scenario_detail
        .groupby("Measure_Family")[
            "Scenario_Value"
        ]
        .rank(
            ascending=False,
            method="average"
        )
    )

    # -----------------------------------------------------
    # Composite standardisation
    # -----------------------------------------------------

    scenario_detail[
        "Standardised_Signal_0_100"
    ] = (
        scenario_detail
        .groupby(
            "Measure_Family",
            group_keys=False
        )["Scenario_Value"]
        .apply(min_max_signal)
    )

    # -----------------------------------------------------
    # Apply scenario weights
    # -----------------------------------------------------

    if weight_mapping is None:

        # Equal weighting across all indicators included
        number_of_families = (
            scenario_detail[
                "Measure_Family"
            ].nunique()
        )

        scenario_detail[
            "Scenario_Weight"
        ] = (
            1 / number_of_families
        )

    else:

        scenario_detail[
            "Scenario_Weight"
        ] = (
            scenario_detail[
                "Measure_Family"
            ].map(weight_mapping)
        )

        if scenario_detail[
            "Scenario_Weight"
        ].isna().any():

            missing_weights = (
                scenario_detail.loc[
                    scenario_detail[
                        "Scenario_Weight"
                    ].isna(),
                    "Measure_Family"
                ]
                .unique()
                .tolist()
            )

            raise ValueError(
                "Weights are missing for: "
                f"{missing_weights}"
            )

    # Check that weights total 1 for each specialty
    weight_check = (
        scenario_detail
        .groupby("Specialty")[
            "Scenario_Weight"
        ]
        .sum()
    )

    if not np.allclose(
        weight_check.values,
        1.0
    ):
        raise ValueError(
            f"Weights do not total 1 in "
            f"scenario '{scenario_name}':\n"
            f"{weight_check}"
        )

    scenario_detail[
        "Weighted_Contribution"
    ] = (
        scenario_detail[
            "Standardised_Signal_0_100"
        ]
        * scenario_detail[
            "Scenario_Weight"
        ]
    )

    # -----------------------------------------------------
    # Create specialty summary
    # -----------------------------------------------------

    scenario_summary = (
        scenario_detail
        .groupby(
            "Specialty",
            as_index=False
        )
        .agg(
            Mean_Rank=(
                "Indicator_Rank",
                "mean"
            ),
            Composite_Signal_0_100=(
                "Weighted_Contribution",
                "sum"
            ),
            Indicators_Included=(
                "Measure_Family",
                "nunique"
            ),
            Weight_Total=(
                "Scenario_Weight",
                "sum"
            )
        )
    )

    scenario_summary[
        "Rank_Method_Position"
    ] = (
        scenario_summary[
            "Mean_Rank"
        ]
        .rank(
            ascending=True,
            method="min"
        )
    )

    scenario_summary[
        "Composite_Position"
    ] = (
        scenario_summary[
            "Composite_Signal_0_100"
        ]
        .rank(
            ascending=False,
            method="min"
        )
    )

    scenario_summary[
        "Scenario"
    ] = scenario_name

    scenario_summary[
        "Aggregation"
    ] = value_column

    scenario_detail[
        "Scenario"
    ] = scenario_name

    return (
        scenario_detail,
        scenario_summary
    )

#### Mean vs median

In [182]:
median_detail, median_summary = (
    run_signal_scenario(
        specialty_inputs=(
            specialty_indicator_summary
        ),
        scenario_name=(
            "Median ICB values - equal indicator weights"
        ),
        value_column="ICB_Median"
    )
)

display(
    median_summary
    .sort_values(
        "Composite_Position"
    )
    .round(2)
)

,Specialty,Mean_Rank,Composite_Signal_0_100,Indicators_Included,Weight_Total,Rank_Method_Position,Composite_Position,Scenario,Aggregation
3,Trauma and Orthopaedics,1.50,83.71,4,1.0,1.0,1.0,Median ICB values - equal indicator weights,ICB_Median
0,Cardiovascular,2.75,46.81,4,1.0,2.0,2.0,Median ICB values - equal indicator weights,ICB_Median
2,Respiratory,2.75,46.74,4,1.0,2.0,3.0,Median ICB values - equal indicator weights,ICB_Median
1,Gynaecology,3.00,40.80,4,1.0,4.0,4.0,Median ICB values - equal indicator weights,ICB_Median


In [183]:
mean_detail, mean_summary = (
    run_signal_scenario(
        specialty_inputs=(
            specialty_indicator_summary
        ),
        scenario_name=(
            "Mean ICB values - equal indicator weights"
        ),
        value_column="ICB_Mean"
    )
)

display(
    mean_summary
    .sort_values(
        "Composite_Position"
    )
    .round(2)
)

,Specialty,Mean_Rank,Composite_Signal_0_100,Indicators_Included,Weight_Total,Rank_Method_Position,Composite_Position,Scenario,Aggregation
3,Trauma and Orthopaedics,1.50,83.08,4,1.0,1.0,1.0,Mean ICB values - equal indicator weights,ICB_Mean
2,Respiratory,2.75,47.69,4,1.0,2.0,2.0,Mean ICB values - equal indicator weights,ICB_Mean
0,Cardiovascular,2.75,45.66,4,1.0,2.0,3.0,Mean ICB values - equal indicator weights,ICB_Mean
1,Gynaecology,3.00,35.43,4,1.0,4.0,4.0,Mean ICB values - equal indicator weights,ICB_Mean


In [184]:
mean_median_comparison = (
    median_summary[
        [
            "Specialty",
            "Rank_Method_Position",
            "Composite_Position",
            "Composite_Signal_0_100"
        ]
    ]
    .rename(
        columns={
            "Rank_Method_Position":
                "Median_Rank_Position",
            "Composite_Position":
                "Median_Composite_Position",
            "Composite_Signal_0_100":
                "Median_Composite_Signal"
        }
    )
    .merge(
        mean_summary[
            [
                "Specialty",
                "Rank_Method_Position",
                "Composite_Position",
                "Composite_Signal_0_100"
            ]
        ]
        .rename(
            columns={
                "Rank_Method_Position":
                    "Mean_Rank_Position",
                "Composite_Position":
                    "Mean_Composite_Position",
                "Composite_Signal_0_100":
                    "Mean_Composite_Signal"
            }
        ),
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
)

mean_median_comparison[
    "Composite_Position_Changed"
] = (
    mean_median_comparison[
        "Median_Composite_Position"
    ]
    .ne(
        mean_median_comparison[
            "Mean_Composite_Position"
        ]
    )
)

display(
    mean_median_comparison.round(2)
)

,Specialty,Median_Rank_Position,Median_Composite_Position,Median_Composite_Signal,Mean_Rank_Position,Mean_Composite_Position,Mean_Composite_Signal,Composite_Position_Changed
0,Cardiovascular,2.0,2.0,46.81,2.0,3.0,45.66,True
1,Gynaecology,4.0,4.0,40.80,4.0,4.0,35.43,False
2,Respiratory,2.0,3.0,46.74,2.0,2.0,47.69,True
3,Trauma and Orthopaedics,1.0,1.0,83.71,1.0,1.0,83.08,False


#### Equal indicator vs equal domain weighting
because we have 2x hospital burden indicators, 1x scale of pressure, 1x access pressure

In [185]:
domain_equal_weights = {
    "RTT_18plus": 1 / 3,
    "RTT_Incomplete": 1 / 3,
    "Emergency_LOS_6plus": 1 / 6,
    "Elective_LOS_6plus": 1 / 6
}

In [186]:
domain_weight_detail, domain_weight_summary = (
    run_signal_scenario(
        specialty_inputs=(
            specialty_indicator_summary
        ),
        scenario_name=(
            "Median ICB values - equal domain weights"
        ),
        value_column="ICB_Median",
        weight_mapping=domain_equal_weights
    )
)

display(
    domain_weight_summary
    .sort_values(
        "Composite_Position"
    )
    .round(2)
)

,Specialty,Mean_Rank,Composite_Signal_0_100,Indicators_Included,Weight_Total,Rank_Method_Position,Composite_Position,Scenario,Aggregation
3,Trauma and Orthopaedics,1.50,89.14,4,1.0,1.0,1.0,Median ICB values - equal domain weights,ICB_Median
1,Gynaecology,3.00,54.40,4,1.0,4.0,2.0,Median ICB values - equal domain weights,ICB_Median
0,Cardiovascular,2.75,42.23,4,1.0,2.0,3.0,Median ICB values - equal domain weights,ICB_Median
2,Respiratory,2.75,31.16,4,1.0,2.0,4.0,Median ICB values - equal domain weights,ICB_Median


In [187]:
#compare with the baseline

weighting_comparison = (
    median_summary[
        [
            "Specialty",
            "Composite_Position",
            "Composite_Signal_0_100"
        ]
    ]
    .rename(
        columns={
            "Composite_Position":
                "Equal_Indicator_Position",
            "Composite_Signal_0_100":
                "Equal_Indicator_Signal"
        }
    )
    .merge(
        domain_weight_summary[
            [
                "Specialty",
                "Composite_Position",
                "Composite_Signal_0_100"
            ]
        ]
        .rename(
            columns={
                "Composite_Position":
                    "Equal_Domain_Position",
                "Composite_Signal_0_100":
                    "Equal_Domain_Signal"
            }
        ),
        on="Specialty",
        how="outer",
        validate="one_to_one"
    )
)

weighting_comparison[
    "Position_Changed"
] = (
    weighting_comparison[
        "Equal_Indicator_Position"
    ]
    .ne(
        weighting_comparison[
            "Equal_Domain_Position"
        ]
    )
)

display(
    weighting_comparison.round(2)
)

,Specialty,Equal_Indicator_Position,Equal_Indicator_Signal,Equal_Domain_Position,Equal_Domain_Signal,Position_Changed
0,Cardiovascular,2.0,46.81,3.0,42.23,True
1,Gynaecology,4.0,40.80,2.0,54.40,True
2,Respiratory,3.0,46.74,4.0,31.16,True
3,Trauma and Orthopaedics,1.0,83.71,1.0,89.14,False


#### Leave one indicator out - testing
Does the result still hold if any one indicator is removed?

In [188]:
measure_families = (
    specialty_indicator_summary[
        "Measure_Family"
    ]
    .dropna()
    .unique()
    .tolist()
)

print(measure_families)

['RTT_18plus', 'Elective_LOS_6plus', 'Emergency_LOS_6plus', 'RTT_Incomplete']


In [189]:
leave_one_out_summaries = []
leave_one_out_details = {}

for measure_family in measure_families:

    scenario_name = (
        f"Exclude {measure_family}"
    )

    scenario_detail, scenario_summary = (
        run_signal_scenario(
            specialty_inputs=(
                specialty_indicator_summary
            ),
            scenario_name=scenario_name,
            value_column="ICB_Median",
            excluded_measure_family=(
                measure_family
            )
        )
    )

    leave_one_out_details[
        measure_family
    ] = scenario_detail

    leave_one_out_summaries.append(
        scenario_summary
    )

leave_one_out_summary = pd.concat(
    leave_one_out_summaries,
    ignore_index=True
)

display(
    leave_one_out_summary[
        [
            "Scenario",
            "Specialty",
            "Composite_Signal_0_100",
            "Composite_Position",
            "Mean_Rank",
            "Rank_Method_Position",
            "Indicators_Included"
        ]
    ]
    .sort_values(
        [
            "Scenario",
            "Composite_Position"
        ]
    )
    .round(2)
)

,Scenario,Specialty,Composite_Signal_0_100,Composite_Position,Mean_Rank,Rank_Method_Position,Indicators_Included
7,Exclude Elective_LOS_6plus,Trauma and Orthopaedics,100.00,1.0,1.00,1.0,3
5,Exclude Elective_LOS_6plus,Gynaecology,54.40,2.0,2.67,2.0,3
4,Exclude Elective_LOS_6plus,Cardiovascular,41.64,3.0,3.00,3.0,3
6,Exclude Elective_LOS_6plus,Respiratory,28.99,4.0,3.33,4.0,3
11,Exclude Emergency_LOS_6plus,Trauma and Orthopaedics,78.29,1.0,1.67,1.0,3
9,Exclude Emergency_LOS_6plus,Gynaecology,54.40,2.0,2.67,2.0,3
8,Exclude Emergency_LOS_6plus,Cardiovascular,42.83,3.0,2.67,2.0,3
10,Exclude Emergency_LOS_6plus,Respiratory,33.33,4.0,3.00,4.0,3
3,Exclude RTT_18plus,Trauma and Orthopaedics,78.29,1.0,1.67,1.0,3
2,Exclude RTT_18plus,Respiratory,62.32,2.0,2.33,2.0,3


In [190]:
leave_one_out_position_table = (
    leave_one_out_summary
    .pivot(
        index="Specialty",
        columns="Scenario",
        values="Composite_Position"
    )
    .reset_index()
)

display(
    leave_one_out_position_table
)

Scenario,Specialty,Exclude Elective_LOS_6plus,Exclude Emergency_LOS_6plus,Exclude RTT_18plus,Exclude RTT_Incomplete
0,Cardiovascular,3.0,3.0,3.0,3.0
1,Gynaecology,2.0,2.0,4.0,4.0
2,Respiratory,4.0,4.0,2.0,2.0
3,Trauma and Orthopaedics,1.0,1.0,1.0,1.0


In [191]:
leave_one_out_score_table = (
    leave_one_out_summary
    .pivot(
        index="Specialty",
        columns="Scenario",
        values="Composite_Signal_0_100"
    )
    .reset_index()
)

display(
    leave_one_out_score_table
    .round(2)
)

Scenario,Specialty,Exclude Elective_LOS_6plus,Exclude Emergency_LOS_6plus,Exclude RTT_18plus,Exclude RTT_Incomplete
0,Cardiovascular,41.64,42.83,54.96,47.80
1,Gynaecology,54.40,54.40,22.34,32.06
2,Respiratory,28.99,33.33,62.32,62.32
3,Trauma and Orthopaedics,100.00,78.29,78.29,78.29


## Combined sensitivity summary

In [192]:
sensitivity_summary = pd.concat(
    [
        median_summary,
        mean_summary,
        domain_weight_summary,
        leave_one_out_summary
    ],
    ignore_index=True
)

sensitivity_summary = (
    sensitivity_summary[
        [
            "Scenario",
            "Specialty",
            "Aggregation",
            "Indicators_Included",
            "Mean_Rank",
            "Rank_Method_Position",
            "Composite_Signal_0_100",
            "Composite_Position"
        ]
    ]
    .sort_values(
        [
            "Scenario",
            "Composite_Position"
        ]
    )
    .reset_index(drop=True)
)

display(
    sensitivity_summary.round(2)
)

,Scenario,Specialty,Aggregation,Indicators_Included,Mean_Rank,Rank_Method_Position,Composite_Signal_0_100,Composite_Position
0,Exclude Elective_LOS_6plus,Trauma and Orthopaedics,ICB_Median,3,1.00,1.0,100.00,1.0
1,Exclude Elective_LOS_6plus,Gynaecology,ICB_Median,3,2.67,2.0,54.40,2.0
2,Exclude Elective_LOS_6plus,Cardiovascular,ICB_Median,3,3.00,3.0,41.64,3.0
3,Exclude Elective_LOS_6plus,Respiratory,ICB_Median,3,3.33,4.0,28.99,4.0
4,Exclude Emergency_LOS_6plus,Trauma and Orthopaedics,ICB_Median,3,1.67,1.0,78.29,1.0
5,Exclude Emergency_LOS_6plus,Gynaecology,ICB_Median,3,2.67,2.0,54.40,2.0
6,Exclude Emergency_LOS_6plus,Cardiovascular,ICB_Median,3,2.67,2.0,42.83,3.0
7,Exclude Emergency_LOS_6plus,Respiratory,ICB_Median,3,3.00,4.0,33.33,4.0
8,Exclude RTT_18plus,Trauma and Orthopaedics,ICB_Median,3,1.67,1.0,78.29,1.0
9,Exclude RTT_18plus,Respiratory,ICB_Median,3,2.33,2.0,62.32,2.0


In [193]:
scenario_leaders = (
    sensitivity_summary
    .loc[
        sensitivity_summary[
            "Composite_Position"
        ].eq(1)
    ]
    [
        [
            "Scenario",
            "Specialty",
            "Composite_Signal_0_100"
        ]
    ]
    .sort_values("Scenario")
    .reset_index(drop=True)
)

display(
    scenario_leaders.round(2)
)

,Scenario,Specialty,Composite_Signal_0_100
0,Exclude Elective_LOS_6plus,Trauma and Orthopaedics,100.00
1,Exclude Emergency_LOS_6plus,Trauma and Orthopaedics,78.29
2,Exclude RTT_18plus,Trauma and Orthopaedics,78.29
3,Exclude RTT_Incomplete,Trauma and Orthopaedics,78.29
4,Mean ICB values - equal indicator weights,Trauma and Orthopaedics,83.08
5,Median ICB values - equal domain weights,Trauma and Orthopaedics,89.14
6,Median ICB values - equal indicator weights,Trauma and Orthopaedics,83.71


# Dashboard Export Preparation
Purpose: Create Power BI-ready datasets.
One row per:
ICB + Specialty + Indicator
This structure supports:
- Tables
- Maps
- Slicers
- Domain views
- Specialty views
- Indicator views


In [197]:
# DATASET 1
# Long dashboard table

dashboard_long = (
    signal_long[
        [
            "ICB_Geography_Code",   # E-code
            "ICB_Code",             # Q-code
            "ICB_Name",
            "Population_2024",
            "IMD_Average_Score",

            "Specialty",
            "Domain",
            "Measure_Family",
            "Indicator",

            "Analysis_Value",
            "Analysis_Unit"
        ]
    ]
    .copy()
)

print(
    "Dashboard long rows:",
    len(dashboard_long)
)

display(
    dashboard_long.head()
)

Dashboard long rows: 672


,ICB_Geography_Code,ICB_Code,ICB_Name,Population_2024,IMD_Average_Score,Specialty,Domain,Measure_Family,Indicator,Analysis_Value,Analysis_Unit
0,E54000040,QOX,"Bath And North East Somerset, Swindon And Wilt...",974918,14.281,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,43.27,Percentage
1,E54000024,QHG,"Bedfordshire, Luton And Milton Keynes ICB",1063247,19.537,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,47.93,Percentage
2,E54000055,QHL,Birmingham And Solihull ICB,1404860,34.770,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,33.14,Percentage
3,E54000062,QUA,Black Country ICB,1262719,30.214,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,47.46,Percentage
4,E54000039,QUY,"Bristol, North Somerset And South Gloucestersh...",1025309,17.706,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,35.21,Percentage


In [199]:
# Dataset 2 - useful for cards and ranking

dashboard_specialty_summary = (
    specialty_indicator_summary.copy()
)

display(
    dashboard_specialty_summary.head()
)

,Specialty,Domain,Measure_Family,Indicator,Specialty_Value,ICB_Mean,ICB_Median,ICB_P10,ICB_P90,ICBs_Available,P90_P10_Variation
0,Cardiovascular,Access Pressure,RTT_18plus,Cardiology_RTT_18plus,38.60000,37.520952,38.60000,26.447000,47.865000,42,21.41800
1,Cardiovascular,Hospital Burden,Elective_LOS_6plus,Cardiology_ElectiveAdmissions_6day+,10.45000,12.214286,10.45000,3.010000,24.810000,42,21.80000
2,Cardiovascular,Hospital Burden,Emergency_LOS_6plus,Cardiology_EmergencyAdmissions_6day+,26.85000,26.883333,26.85000,18.330000,34.510000,42,16.18000
3,Cardiovascular,Scale of Pressure,RTT_Incomplete,Cardiology_RTT_Incomplete,674.29359,693.360994,674.29359,373.106593,1027.970883,42,654.86429
4,Gynaecology,Access Pressure,RTT_18plus,Gynaecology_RTT_18plus,45.35500,44.348571,45.35500,35.294000,52.059000,42,16.76500


In [200]:
# Dataset 3 - useful for top specialty, bar chart, ranking

dashboard_composite = (
    composite_specialty_summary.copy()
)

display(
    dashboard_composite
)

,Specialty,Composite_Signal_0_100,Indicators_Included,Weight_Total,Composite_Rank
3,Trauma and Orthopaedics,89.141021,4,1.0,1.0
1,Gynaecology,54.396368,4,1.0,2.0
0,Cardiovascular,42.235861,4,1.0,3.0
2,Respiratory,31.166519,4,1.0,4.0


In [201]:
#dataset 4 = ICB lookup table

dashboard_icb_lookup = (
    signal_long[
        [
            "ICB_Geography_Code",
            "ICB_Code",
            "ICB_Name",
            "Population_2024",
            "IMD_Average_Score"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(
    dashboard_icb_lookup.head()
)

,ICB_Geography_Code,ICB_Code,ICB_Name,Population_2024,IMD_Average_Score
0,E54000040,QOX,"Bath And North East Somerset, Swindon And Wilt...",974918,14.281
1,E54000024,QHG,"Bedfordshire, Luton And Milton Keynes ICB",1063247,19.537
2,E54000055,QHL,Birmingham And Solihull ICB,1404860,34.770
3,E54000062,QUA,Black Country ICB,1262719,30.214
4,E54000039,QUY,"Bristol, North Somerset And South Gloucestersh...",1025309,17.706


## Final Excel Export

In [202]:
with pd.ExcelWriter(
    METHODS_OUTPUT_DIR
    / "Specialty_Signal_Methods_Test.xlsx",
    engine="openpyxl"
) as writer:

    qa_report.to_excel(
        writer,
        sheet_name="QA",
        index=False
    )

    signal_long.to_excel(
        writer,
        sheet_name="ICB Indicator Data",
        index=False
    )

    specialty_indicator_summary.to_excel(
        writer,
        sheet_name="Specialty Inputs",
        index=False
    )

    rank_results.to_excel(
        writer,
        sheet_name="Rank Detail",
        index=False
    )

    rank_specialty_summary.to_excel(
        writer,
        sheet_name="Rank Summary",
        index=False
    )

    composite_results.to_excel(
        writer,
        sheet_name="Composite Detail",
        index=False
    )

    composite_specialty_summary.to_excel(
        writer,
        sheet_name="Composite Summary",
        index=False
    )

    method_comparison.to_excel(
        writer,
        sheet_name="Method Comparison",
        index=False
    )

    sensitivity_summary.to_excel(
        writer,
        sheet_name="Sensitivity Summary",
        index=False
    )

    scenario_leaders.to_excel(
        writer,
        sheet_name="Scenario Leaders",
        index=False
    )

    pca_missing_summary.to_excel(
        writer,
        sheet_name="PCA Missingness",
        index=False
    )

    pca_imputation_report.to_excel(
        writer,
        sheet_name="PCA Imputation",
        index=False
    )

    pca_standardised_output.to_excel(
        writer,
        sheet_name="PCA Standardised Data",
        index=False
    )

    pca_explained_variance.to_excel(
        writer,
        sheet_name="PCA Explained Variance",
        index=False
    )

    pca_loadings.to_excel(
        writer,
        sheet_name="PCA Loadings",
        index=False
    )

    pca_scores.to_excel(
        writer,
        sheet_name="PCA ICB Scores",
        index=False
    )

    pca_specialty_summary.to_excel(
        writer,
        sheet_name="PCA Specialty Summary",
        index=False
    )

    dashboard_long.to_excel(
        writer,
        sheet_name="Dashboard_Long",
        index=False
    )

    dashboard_specialty_summary.to_excel(
        writer,
        sheet_name="Dashboard_Specialty",
        index=False
    )

    dashboard_composite.to_excel(
        writer,
        sheet_name="Dashboard_Composite",
        index=False
    )

    dashboard_icb_lookup.to_excel(
        writer,
        sheet_name="Dashboard_ICB_Lookup",
        index=False
    )

print(
    "Methods test workbook saved to:"
)

print(
    METHODS_OUTPUT_DIR
    / "Specialty_Signal_Methods_Test.xlsx"
)

Methods test workbook saved to:
..\outputs\Methods_Test\Specialty_Signal_Methods_Test.xlsx
